# Step-by-Step Pipeline Template

This notebook walks through a full modeling run from raw data to staged fitting, uncertainty, manifest saving, and resume workflow.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using GrowthParameterEstimation
using DataFrames, CSV, Random
println("Loaded GrowthParameterEstimation")

## 1) Load data

Replace the CSV path with your real dataset.

In [ ]:
raw = CSV.read("path/to/your_dataset.csv", DataFrame)
df = normalize_schema(raw)
first(df, 5)

## 2) Validate schema and metadata early

Strict mode catches missing metadata before expensive fitting runs.

In [ ]:
validate_timeseries(df)
validate_required_metadata(df)
validate_strict_schema(df)

## 3) Generate QC report

In [ ]:
qc = generate_qc_report(df)
qc_paths = save_qc_report(qc; output_dir = "results/template_qc")
println(qc_paths)
first(qc.condition_summary, 10)

## 4) Choose staged template

This template enforces:
- population-global `r`,`K` for naive vs resistant
- per-cell-line `ic50` learned in treated monoculture and inherited downstream

In [ ]:
stages = default_population_cellline_stages(df; populations=["naive", "resistant"])
println("Number of stages: ", length(stages))
for s in stages[1:min(8, length(stages))]
    println(s.name)
end

## 5) Run staged pipeline (auto mode)

In [ ]:
cfg = default_config(output_dir = "results/template_staged_auto")
run_auto = run_staged_pipeline(
    df;
    stages = stages,
    config = cfg,
    selection_mode = :best_bic,
    strict_schema = true,
    qc_before_fit = true,
    n_bootstrap = 20,
    export_stage_results = true,
)
println("Completed: ", run_auto.completed)
println("Manifest: ", run_auto.manifest_path)

## 6) Run staged pipeline (checkpoint/manual mode)

Use this when you want human review of top models at each stage.

In [ ]:
manual_choices = Dict(
    "untreated_monoculture_naive" => "logistic_growth",
    "untreated_monoculture_resistant" => "gompertz_growth",
)
run_checkpoint = run_staged_pipeline(
    df;
    stages = stages,
    config = default_config(output_dir = "results/template_staged_checkpoint"),
    selection_mode = :manual,
    manual_choices = manual_choices,
    strict_schema = true,
    n_bootstrap = 20,
    export_stage_results = true,
)
println("Halted stage: ", run_checkpoint.halted_stage)
println("Manifest: ", run_checkpoint.manifest_path)

## 7) Resume from a halted stage

In [ ]:
resume_manifest = load_run_manifest(run_checkpoint.manifest_path)
println("Completed stages from manifest: ", resume_manifest.completed_stages)

# Add model decisions for later stages here:
resume_choices = Dict(
    "untreated_monoculture_naive" => "logistic_growth",
    "untreated_monoculture_resistant" => "gompertz_growth",
    # Example: "treated_monoculture_naive_a549" => "theta_logistic_hill_inhibition",
)

run_resumed = run_staged_pipeline(
    df;
    stages = stages,
    config = default_config(output_dir = "results/template_staged_checkpoint"),
    selection_mode = :manual,
    manual_choices = resume_choices,
    strict_schema = true,
    resume_manifest_path = run_checkpoint.manifest_path,
    resume_from_stage = run_checkpoint.halted_stage,
    export_stage_results = true,
)
println("Completed after resume: ", run_resumed.completed)